In [1]:
import pyspark
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql import types

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.ui.port", "4040") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

25/03/05 15:36:55 WARN Utils: Your hostname, DESKTOP-QRKC0LG resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/03/05 15:36:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/05 15:36:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

In [ ]:
!gunzip -k fhvhv_tripdata_2021-01.csv.gz

In [ ]:
!wc -l fhvhv_tripdata_2021-01.csv

In [ ]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [ ]:
df.schema

In [ ]:
df.show()

In [ ]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [ ]:
df_pandas = pd.read_csv('head.csv')

In [ ]:
df_pandas.dtypes

In [ ]:
df_pandas.head()

In [ ]:
#!pip install --upgrade setuptools

In [ ]:
spark.createDataFrame(df_pandas).schema

In [ ]:
df_pandas.info()  # Check for any columns with mixed data types


In [ ]:
spark_df = spark.createDataFrame(df_pandas)
spark_df.printSchema()


In [4]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [5]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [6]:
df = df.repartition(24)

In [7]:
df.write.parquet('fhvhv/2021/01/')


In [10]:
df = spark.read.parquet('fhvhv/2021/01/')

In [12]:

df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [13]:
from pyspark.sql import functions as F

In [14]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02875|2021-01-01 05:01:08|2021-01-01 05:10:11|          91|          89|   NULL|
|           HV0003|              B02867|2021-01-01 09:27:50|2021-01-01 09:46:50|         162|         106|   NULL|
|           HV0003|              B02880|2021-01-01 07:25:16|2021-01-01 07:50:46|          61|          76|   NULL|
|           HV0003|              B02878|2021-01-01 05:00:55|2021-01-01 05:10:00|         213|         147|   NULL|
|           HV0003|              B02875|2021-01-01 10:34:32|2021-01-01 10:52:47|          17|         133|   NULL|
|           HV0003|              B02876|2021-01-01 13:44:14|2021-01-01 13:56:07|

In [15]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [16]:
crazy_stuff('B02884')

's/b44'

In [17]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())


In [18]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/b3b| 2021-01-01|  2021-01-01|          91|          89|
|  e/b33| 2021-01-01|  2021-01-01|         162|         106|
|  a/b40| 2021-01-01|  2021-01-01|          61|          76|
|  e/b3e| 2021-01-01|  2021-01-01|         213|         147|
|  e/b3b| 2021-01-01|  2021-01-01|          17|         133|
|  e/b3c| 2021-01-01|  2021-01-01|          60|          51|
|  e/b14| 2021-01-01|  2021-01-01|          13|         148|
|  e/acc| 2021-01-01|  2021-01-01|         229|          73|
|  e/b3b| 2021-01-01|  2021-01-01|          49|          61|
|  e/9ce| 2021-01-01|  2021-01-01|          81|          51|
|  e/9ce| 2021-01-01|  2021-01-01|          79|         225|
|  e/b30| 2021-01-01|  2021-01-01|         142|         208|
|  e/9ce| 2021-01-01|  2021-01-01|         181|          89|
|  e/acc| 2021-01-01|  2

In [19]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [20]:
!head -n 10 head.csv